# tinyLMTune — Question Answering

This notebook demonstrates 3 ways to train TinyBERT for **qna** using tinyLMTune:

1. **Synthetic data** — auto-generated via Flan-T5/Mistral
2. **Benchmark data** — real HuggingFace dataset (SQuAD)
3. **Raw user data** — your own text, structured or unstructured

Each example runs the full pipeline: data → token analysis → search space recommendation → GA optimisation → model save → inference.

## Setup

In [1]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

# Install if needed (uncomment):
!pip install -e ../../tinylmtune_v2/

from tinylmtune import optimize_slm, TinyInference, print_token_analysis, print_recommendation

Obtaining file:///home/sagemaker-user/SLM/tinylmtune_v2
  Preparing metadata (setup.py) ... done
  Attempting uninstall: tinylmtune
    Found existing installation: tinylmtune 2.0.0
    Uninstalling tinylmtune-2.0.0:
      Successfully uninstalled tinylmtune-2.0.0
  DEPRECATION: Legacy editable install of tinylmtune==2.0.0 from file:///home/sagemaker-user/SLM/tinylmtune_v2 (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for tinylmtune


2026-05-18 06:42:09,047 | datasets | PyTorch version 2.4.1.post300 available.
2026-05-18 06:42:09,049 | datasets | TensorFlow version 2.17.0 available.
2026-05-18 06:42:10.069024: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-18 06:42:10.081349: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-18 06:42:10.085896: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-18 06:42:10.095517: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following in

---
## Example 1 — Synthetic Data (via Flan-T5)

No data needed. Flan-T5/Mistral generates training data from a topic prompt.

**Requirements:** Flan-T5 must be installed and running (`pip install sentencepiece`), with `mistral` model pulled (``).

In [11]:
best = optimize_slm(
    task="qna",
    corpus_prompt="Generate question-answer-context samples about world history and geography",
    n_examples=1000,
    max_len=8,
    pop_size=4,
    generations=2,
    output_dir="models/qna_synthetic",
)
print("Best config:", best)

2026-05-18 07:00:16,418 | tinylmtune._internal.pipeline | Model will be saved to: /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/qna_synthetic
2026-05-18 07:00:16,418 | tinylmtune._internal.pipeline | No user data — generating synthetic corpus
2026-05-18 07:00:16,419 | tinylmtune._internal.corpus_gen | Generating 1000 qna examples about 'Generate question-answer-context samples about world history and geography' using google/flan-t5-small ...
2026-05-18 07:00:16,419 | tinylmtune._internal.llm_backend | Loading LLM: google/flan-t5-small ...
2026-05-18 07:00:16,733 | tinylmtune._internal.llm_backend | LLM loaded on cuda (76961152 params)
2026-05-18 07:00:35,598 | tinylmtune._internal.corpus_gen | Generated 50 / 1000 examples (50 valid so far)
2026-05-18 07:00:53,639 | tinylmtune._internal.corpus_gen | Generated 100 / 1000 examples (96 valid so far)
2026-05-18 07:01:12,631 | tinylmtune._internal.corpus_gen | Generated 150 / 1000 examples (144 valid so far)
2026-05-18 07:01:33,633

Epoch,Training Loss,Validation Loss,Exact Match,Start Acc,End Acc
1,2.206600,1.985364,0.306122,0.551020,0.352041
2,1.875300,1.910932,0.301020,0.551020,0.357143
3,1.628800,1.963242,0.295918,0.540816,0.357143


2026-05-18 07:07:00,185 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 1.985364317893982, 'eval_exact_match': 0.30612244897959184, 'eval_start_acc': 0.5510204081632653, 'eval_end_acc': 0.3520408163265306, 'eval_runtime': 0.1826, 'eval_samples_per_second': 1073.227, 'eval_steps_per_second': 268.307, 'epoch': 3.0}
Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-05-18 07:07:00,410 | tinylmtune._internal.trainer | Training: lr=0.0001201671422284161 bs=4 epochs=10 dropout=0.11 attn_drop=0.04 grad_accum=4 scheduler=linear label_smooth=0.076 grad_norm=1.2


Epoch,Training Loss,Validation Loss,Exact Match,Start Acc,End Acc
1,2.525200,1.912158,0.270408,0.551020,0.316327
2,1.791300,1.804998,0.311224,0.581633,0.352041
3,1.610300,1.927438,0.346939,0.581633,0.377551
4,1.435800,1.814174,0.367347,0.586735,0.397959
5,1.263500,1.916751,0.418367,0.581633,0.454082
6,1.090300,1.927667,0.408163,0.576531,0.469388
7,0.981900,2.011240,0.341837,0.556122,0.433673


2026-05-18 07:07:19,717 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 1.9167511463165283, 'eval_exact_match': 0.41836734693877553, 'eval_start_acc': 0.5816326530612245, 'eval_end_acc': 0.45408163265306123, 'eval_runtime': 0.1818, 'eval_samples_per_second': 1078.217, 'eval_steps_per_second': 269.554, 'epoch': 7.0}
Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-05-18 07:07:19,937 | tinylmtune._internal.trainer | Training: lr=0.00021419412708598376 bs=16 epochs=4 dropout=0.02 attn_drop=0.08 grad_accum=4 scheduler=cosine_with_restarts label_smooth=0.060 grad_norm=4.1


...

Epoch,Training Loss,Validation Loss,Exact Match,Start Acc,End Acc
1,2.432600,1.901709,0.270408,0.556122,0.331633
2,1.764400,1.742952,0.311224,0.581633,0.367347
3,1.562000,2.020001,0.346939,0.581633,0.377551
4,1.476800,2.023061,0.357143,0.581633,0.387755
5,1.296200,2.125039,0.372449,0.566327,0.408163
6,1.065400,2.099529,0.357143,0.545918,0.423469
7,0.916600,2.312132,0.326531,0.520408,0.403061


2026-05-18 07:09:00,146 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 2.1250393390655518, 'eval_exact_match': 0.37244897959183676, 'eval_start_acc': 0.5663265306122449, 'eval_end_acc': 0.40816326530612246, 'eval_runtime': 0.1796, 'eval_samples_per_second': 1091.34, 'eval_steps_per_second': 272.835, 'epoch': 7.0}
2026-05-18 07:09:00,147 | tinylmtune._internal.ga_optimizer | Gen 2/2 — best=0.4031  avg=0.3865  worst=0.3724
2026-05-18 07:09:00,148 | tinylmtune._internal.pipeline | Best config (fitness=0.4184): {'learning_rate': 0.0001201671422284161, 'batch_size': 4, 'epochs': 10, 'warmup_ratio': 0.06, 'weight_decay': 0.065, 'dropout': 0.109, 'attention_dropout': 0.044, 'gradient_accumulation_steps': 4, 'lr_scheduler_type': 'linear', 'label_smoothing': 0.076, 'max_grad_norm': 1.22, 'fitness': 0.41836734693877553}
Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['qa_outp

Epoch,Training Loss,Validation Loss,Exact Match,Start Acc,End Acc
1,2.507800,1.927080,0.285714,0.556122,0.341837
2,1.798600,1.821059,0.285714,0.571429,0.326531
3,1.627100,1.947816,0.352041,0.586735,0.397959
4,1.476600,1.779512,0.377551,0.581633,0.428571
5,1.383800,2.106715,0.382653,0.591837,0.423469
6,1.356800,2.032934,0.362245,0.540816,0.418367
7,1.206200,2.079716,0.316327,0.535714,0.382653


2026-05-18 07:09:18,286 | tinylmtune._internal.trainer | Eval metrics: {'eval_loss': 2.106714963912964, 'eval_exact_match': 0.3826530612244898, 'eval_start_acc': 0.5918367346938775, 'eval_end_acc': 0.42346938775510207, 'eval_runtime': 0.1897, 'eval_samples_per_second': 1033.175, 'eval_steps_per_second': 258.294, 'epoch': 7.0}
2026-05-18 07:09:19,103 | tinylmtune._internal.inference | Model saved → /home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/qna_synthetic


Best config: {'learning_rate': 0.0001201671422284161, 'batch_size': 4, 'epochs': 10, 'warmup_ratio': 0.06, 'weight_decay': 0.065, 'dropout': 0.109, 'attention_dropout': 0.044, 'gradient_accumulation_steps': 4, 'lr_scheduler_type': 'linear', 'label_smoothing': 0.076, 'max_grad_norm': 1.22, 'fitness': 0.41836734693877553, 'max_len': 24, 'task': 'qna', 'output_dir': '/home/sagemaker-user/SLM/tinylmtune_v2/notebooks/models/qna_synthetic', 'ga_history': [{'generation': 1, 'individual': 0, 'fitness': 0.30612244897959184, 'learning_rate': 0.00032151626523665245, 'batch_size': 4, 'epochs': 6, 'warmup_ratio': 0.073, 'weight_decay': 0.014, 'dropout': 0.02, 'attention_dropout': 0.148, 'gradient_accumulation_steps': 1, 'lr_scheduler_type': 'constant_with_warmup', 'label_smoothing': 0.003, 'max_grad_norm': 0.92}, {'generation': 1, 'individual': 1, 'fitness': 0.41836734693877553, 'learning_rate': 0.0001201671422284161, 'batch_size': 4, 'epochs': 10, 'warmup_ratio': 0.06, 'weight_decay': 0.065, 'drop

### Inference on synthetic model

In [17]:
model = TinyInference("models/qna_synthetic")
result = model.predict("What is the capital of France?", context="Paris is capital of France")
print(result)

2026-05-18 07:10:45,745 | tinylmtune._internal.inference | Loaded qna model from models/qna_synthetic
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'answer': 'paris', 'start': 9, 'end': 9}


---
## Example 2 — Benchmark Data (SQuAD)

Uses a real HuggingFace dataset. No Flan-T5 needed.

### Load SQuAD dataset

In [18]:
from datasets import load_dataset
ds = load_dataset("squad", split="train")
ds = ds.shuffle(seed=42).select(range(5000))
benchmark_data = []
for r in ds:
    answer = r["answers"]["text"][0] if r["answers"]["text"] else ""
    if not answer:
        continue
    benchmark_data.append({
        "question": r["question"],
        "context": r["context"],
        "answer": answer,
    })
print(f"Loaded {len(benchmark_data)} records")
print(f"Sample keys: {benchmark_data[0].keys()}")
print(f"Sample Q: {benchmark_data[0]['question']}")
print(f"Sample A: {benchmark_data[0]['answer']}")
print(f"Sample C: {benchmark_data[0]['context'][:100]}...")

Loaded 5000 records
Sample keys: dict_keys(['question', 'context', 'answer'])
Sample Q: What percentage of Egyptians polled support death penalty for those leaving Islam?
Sample A: 84%
Sample C: The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for reli...


In [ ]:
print(benchmark_data[0].keys())
print(benchmark_data[0])

### Analyze token lengths

In [ ]:
print_token_analysis(benchmark_data, task="qna")

### Check recommended search space

In [ ]:
print_recommendation(n_samples=len(benchmark_data), task="qna")

In [ ]:
## Use recommended search space 

GA_SEARCH_SPACE = {'learning_rate': (5e-06, 0.001),
 'batch_size': [4, 8, 16, 32],
 'epochs': (2, 8),
 'warmup_ratio': (0.0, 0.3),
 'weight_decay': (0.0, 0.05),
 'dropout': (0.0, 0.2),
 'attention_dropout': (0.0, 0.2),
 'gradient_accumulation_steps': [1, 2, 4, 8],
 'lr_scheduler_type': ['linear',
  'cosine',
  'cosine_with_restarts',
  'constant_with_warmup'],
 'label_smoothing': (0.0, 0.1),
 'max_grad_norm': (0.5, 5.0)}

### Train with GA optimisation

In [ ]:
best = optimize_slm(
    task="qna",
    user_data=benchmark_data,
    max_len=32,
    pop_size=4,
    generations=2,
    output_dir="models/qna_benchmark",
)
print("Best config:", best)

### Visualize GA Results

5 plots showing how the GA searched for the best hyperparameters:
1. **Fitness progress** — best/avg/worst per generation
2. **Parameter scatter** — each param vs fitness (best = red star)
3. **Scheduler comparison** — box plot by LR scheduler type
4. **Config evolution** — how the best config changed over generations
5. **Population heatmap** — all individuals in the last generation

In [ ]:
from tinylmtune import plot_results, print_best_config_table

# Print formatted best config
print_best_config_table(best)

# Generate all 5 plots
figs = plot_results(best, save_dir="plots/benchmark")

### Inference on benchmark model

In [ ]:
model = TinyInference("models/qna_benchmark")
result = model.predict("Who invented the telephone?", context="Alexander Graham Bell invented the telephone in 1876.")
print(result)

---
## Example 3 — Raw User Data

Three sub-examples showing different input formats:
- **3a.** Structured dicts (correct format)
- **3b.** Raw text strings (auto-labelled via Flan-T5)
- **3c.** Wrong-format dicts (auto-detected and converted)

### 3a. Structured dicts (used directly, no Flan-T5)

In [ ]:
my_data = [
    {"question": "What is Python?", "answer": "A high-level programming language"},
    {"question": "Who created Linux?", "answer": "Linus Torvalds"},
    {"question": "What is TinyBERT?", "answer": "A distilled version of BERT"},
    {"question": "What does GA stand for?", "answer": "Genetic Algorithm"},
    {"question": "What is fine-tuning?", "answer": "Adapting a pretrained model to a specific task"},
    {"question": "What is tokenization?", "answer": "Splitting text into tokens for model input"},
    {"question": "What is NLP?", "answer": "Natural Language Processing"},
    {"question": "What is a transformer?", "answer": "A neural network architecture using attention"},
    {"question": "What is BERT?", "answer": "Bidirectional Encoder Representations from Transformers"},
    {"question": "What is overfitting?", "answer": "When a model memorises training data instead of learning patterns"},
    {"question": "What is dropout?", "answer": "Randomly deactivating neurons during training to prevent overfitting"},
    {"question": "What is a loss function?", "answer": "A function that measures how wrong the model predictions are"},
]

best = optimize_slm(
    task="qna",
    user_data=my_data,
    pop_size=4,
    generations=2,
    output_dir="models/qna_user",
)

### 3b. Raw text strings (requires Flan-T5)

In [ ]:
# Raw text — Flan-T5 generates Q&A pairs from it
raw_texts = [
    "The Eiffel Tower was built in 1889 for the World Fair in Paris.",
    "Water boils at 100 degrees Celsius at sea level.",
    "The human body has 206 bones in the adult skeleton.",
    "DNA stands for deoxyribonucleic acid and carries genetic information.",
    "The speed of light is approximately 300,000 kilometers per second.",
    "Mount Everest is the tallest mountain at 8,849 meters above sea level.",
]

best = optimize_slm(
    task="qna",
    user_data=raw_texts,
    pop_size=4,
    generations=1,
    output_dir="models/qna_raw",
)

### 3c. Wrong-format dicts (requires Flan-T5)

In [ ]:
# Dicts with non-standard keys
wrong_format = [
    {"query": "What year was the moon landing?", "response": "1969"},
    {"query": "Who painted the Mona Lisa?", "response": "Leonardo da Vinci"},
    {"query": "What is the largest ocean?", "response": "Pacific Ocean"},
]

best = optimize_slm(
    task="qna",
    user_data=wrong_format,
    pop_size=4,
    generations=1,
    output_dir="models/qna_wrong",
)

### Visualize user data results

In [ ]:
# Plot results from structured data training (Example 3a)
from tinylmtune import plot_results, print_best_config_table
print_best_config_table(best)
figs = plot_results(best, save_dir="plots/user_data")

### Inference

In [ ]:
model = TinyInference("models/qna_user")
result = model.predict("What is dropout?", context="Dropout randomly deactivates neurons during training.")
print(result)

---
## Summary

| Example | Data source | Flan-T5 needed | Best for |
|---------|-------------|---------------|----------|
| Synthetic | Auto-generated | Yes | Quick prototyping |
| Benchmark | SQuAD | No | Reproducible evaluation |
| User data | Your own text | Depends on format | Production use |

The GA searches 11 hyperparameters: `learning_rate`, `batch_size`, `epochs`, `warmup_ratio`, `weight_decay`, `dropout`, `attention_dropout`, `gradient_accumulation_steps`, `lr_scheduler_type`, `label_smoothing`, `max_grad_norm`.

`max_len` is automatically determined from your data's token length distribution (p95 percentile).